In [1]:
%matplotlib widget
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import riemann_utils.covariances as rutils 
import riemann_utils.plots as rplt
import riemann_utils.matrix_functions as mrtrf

import py_utils.data_managment as dtmn
import py_utils.signal_processing as sgnpr

from pyriemann.utils.distance import distance_riemann
from scipy.io import loadmat

from IPython.display import clear_output

from scripts.processing_centroids import extract_run_centroids, center_covariances
from scripts.processing_oldDataset import processDataset_calibrations, extract_model_centroids
from scripts.processing_newDataset import processDataset_20232024, concatenate_cov_datasets

from importlib import reload
import os
# reload(dtmn)
# reload(rplt)
# reload(sgnpr)
# reload(mrtrf)

In [ ]:
filter_order = 4

windowsLength = 1
windowsShift = 1/16

bandranges = [[8, 26]]
classes = [771, 773]

fs = 512
doNormalize = True

doLogBandPower = False
applyLaplacian = True

pathData = '/home/palatella/workspace/cybathlon_user/'
pathImages = '/home/palatella/workspace/cybathlon_user/images/'

days_modelCreation = ['02/05/2019','09/07/2019','27/10/2020']


if doLogBandPower: print('USING LOG BAND POWER')
if doNormalize: print('USING NORMALIZATION')
if applyLaplacian: print('USING LAPLACIAN')

USING NORMALIZATION
USING LAPLACIAN


In [ ]:
# file_path = f'{pathData}events_labels_dataset_user_20232024.mat'
# proc_data = loadmat(file_path)

# procLabels = dtmn.fix_mat(proc_data['labels'])
# procVector = dtmn.fix_mat(proc_data['procVector'])
# runs_labels = dtmn.fix_mat(proc_data['runs_labels'])
# complete_runs_labels = dtmn.fix_mat(proc_data['complete_runs_labels'])
# validRuns = np.array([x[0] for x in proc_data['validRuns']])
# completeValidRuns = np.array([x[0] for x in proc_data['completeValidRuns']])

# for idx,rLbl in enumerate(runs_labels):
#     if len(rLbl[0]) > 1:
#         t_lbl = np.squeeze(rLbl[0]).astype(float) 
#         t_lbl[t_lbl==1] = 773
#         t_lbl[t_lbl==2] = 771
#         runs_labels[idx][0] = t_lbl

# for idx,rLbl in enumerate(complete_runs_labels):
#     if len(rLbl[0]) > 1:
#         t_lbl = np.squeeze(rLbl[0]).astype(float) 
#         t_lbl[t_lbl==1] = 773
#         t_lbl[t_lbl==2] = 771
#         complete_runs_labels[idx][0] = t_lbl


# channels = np.array(['FC3', 'FC1', 'FCZ', 'FC2', 'FC4', 'C3', 'C1', 'CZ', 'C2', 'C4','CP1', 'CPZ', 'CP2'])

# PSD

In [4]:
import scripts.processing_newDataset as test_module
from importlib import reload
reload(test_module)

<module 'scripts.processing_newDataset' from '/home/palatella/workspace/Riemann-centroids-study/scripts/processing_newDataset.py'>

In [ ]:
filename = 'dataset_user_lhrh_20232024.mat'
# dataset = loadmat(f'{pathData}{filename}')

psdSettings = {}
psdSettings['psdWlength'] = 0.5
psdSettings['wshift'] = 0.0625
psdSettings['mlength'] =  1
psdSettings['pshift'] = 0.25
psdSettings['SampleRate'] = 512

savemat = 'psd_user_lhrh_20232024.mat'

# psd, freq, psd_events, runVector, dayVector, protocolVector, channelBandDict = test_module.processDataset_PSD_user20232024(dataset,  psdSettings['psdWlength'], psdSettings['wshift'], psdSettings['pshift'], mlength=psdSettings['mlength'], fs=psdSettings['SampleRate'])
# dtmn.save(f'{pathData}{savemat}', {'psd': psd, 'freq': freq, 'psd_events': psd_events.to_numpy(), 'column_names': psd_events.columns.values, 'runVector': runVector,  'dayVector': dayVector, 'protocolVector': protocolVector, 'channelBandDict': channelBandDict})

In [ ]:
filename = 'dataset_user_end2024.mat'
dataset = loadmat(f'{pathData}{filename}')

psdSettings = {}
psdSettings['psdWlength'] = 0.5
psdSettings['wshift'] = 0.0625
psdSettings['mlength'] =  1
psdSettings['pshift'] = 0.25
psdSettings['SampleRate'] = 512

savemat = 'psd_user_end2024.mat'

# psd, freq, psd_events, runVector, dayVector, protocolVector, channelBandDict = test_module.processDataset_PSD_user20232024(dataset,  psdSettings['psdWlength'], psdSettings['wshift'], psdSettings['pshift'], mlength=psdSettings['mlength'], fs=psdSettings['SampleRate'])
# dtmn.save(f'{pathData}{savemat}', {'psd': psd, 'freq': freq, 'psd_events': psd_events.to_numpy(), 'column_names': psd_events.columns.values, 'runVector': runVector,  'dayVector': dayVector, 'protocolVector': protocolVector, 'channelBandDict': channelBandDict})

# DATASET 2023 - 2024

In [ ]:
filename = 'dataset_user_lhrh_20232024.mat'
# saveName = 'covs_user_lhrh_20232024.mat'
saveName = 'covs_user_lhrh_20232024_1band_v2.mat'

if doLogBandPower:      saveName = filename.replace('user', 'logBandPower_user')
if not applyLaplacian:  saveName = saveName.replace('user', 'user_noLap')

# processDataset_20232024(pathData, filename, doLogBandPower, bandranges, filter_order, windowsLength, windowsShift, applyLaplacian=applyLaplacian, saveData=True, saveName=saveName, fs=512)

data = loadmat(f'{pathData}{saveName}')
lhrh_classes = [769, 770]
ev = data['cov_events']
columns_name = [x[0] for x in data['column_names'][0]]
col_idx = columns_name.index("typ")
ev[:, col_idx] = np.where(ev[:, col_idx] == lhrh_classes[0], classes[0], ev[:, col_idx])
ev[:, col_idx] = np.where(ev[:, col_idx] == lhrh_classes[1], classes[1], ev[:, col_idx])
data['cov_events'] = ev

covs, cov_events, runs_labels, utilsVector, validRuns, days = concatenate_cov_datasets([], [], [], [], [], [], classes, data)

Concatenating datasets: original dataset is empty. Adapting the new dataset.


In [ ]:
filename = 'dataset_user_end2024.mat'

# saveName = 'covs_user_end2024.mat'
saveName = 'covs_user_end2024_1band_v2.mat'


if doLogBandPower:      saveName = filename.replace('user', 'logBandPower_user')
if not applyLaplacian:  saveName = saveName.replace('user', 'user_noLap')

# processDataset_20232024(pathData, filename, doLogBandPower, bandranges, filter_order, windowsLength, windowsShift, applyLaplacian=applyLaplacian, saveData=True, saveName=saveName, fs=512)

data = loadmat(f'{pathData}{saveName}')
covs, cov_events, runs_labels, utilsVector, validRuns, days = concatenate_cov_datasets(covs, cov_events, runs_labels, utilsVector, validRuns, days, classes, data)

In [ ]:
n_samples = covs.shape[1]
n_days = np.max(cov_events['day'])

dayVector = utilsVector['day']-1
runVector = utilsVector['run']
protocolVector = utilsVector['protocol']
isCFeedback = utilsVector['isCFeedback'].astype(bool)

isCFeedback = isCFeedback & (protocolVector==2)  # only evaluations
validRuns = np.unique(runVector[isCFeedback]).astype(int)

## ---- Comment this part if fisher score and cva
discard_day = '03/10/2023'
n_day_discard = np.where(days == discard_day)[0]
n_run_discard = np.unique(runVector[dayVector==n_day_discard])

validRuns = np.setdiff1d(validRuns, n_run_discard)
## ------------------------------------------------
run_starts = np.unique(runVector, return_index=True)
idx_good_runs = np.intersect1d(run_starts[0], validRuns, return_indices=True)[1]
run_starts = run_starts[1][idx_good_runs]
dates_info = np.unique(dayVector[run_starts], return_index=True)
dates = days[dates_info[0].astype(int)]
x_dates = dates_info[1]


# filenameMatrices = 'centering_logBandPower_matrices_20232024.mat' if doLogBandPower else 'centering_matrices_20232024.mat'
# filenameMatrices = 'centering_logBandPower_matrices.mat' if doLogBandPower else 'centering_matrices.mat'
filenameMatrices = 'centering_matrices_1band_v2.mat'
# filenameMatrices = 'centering_matrices_20232024_1band_.mat' 

if not applyLaplacian: saveName = saveName.replace('matrices', 'matrices_noLap')

if os.path.exists(f'{pathData}{filenameMatrices}'):
    saveTransformMatrices = False
    matrices = loadmat(f'{pathData}{filenameMatrices}')
else:
    print('Creating new centering matrices...')
    saveTransformMatrices = True
    matrices = {}
    matrices['mean_cov'] = None
    matrices['inv_sqrt_mean_cov'] = None

# covs_centered, _, _ = center_covariances(covs, dayVector, isCFeedback, referenceDay=0, saveTransformMatrices=saveTransformMatrices, pathData=pathData, saveName=filenameMatrices, mean_cov=matrices['mean_cov'], inv_sqrt_mean_cov=matrices['inv_sqrt_mean_cov'])

# saveName = 'run_centroids_user_20232024.mat'
# saveName = 'run_centroids_user_20232024_20192020ref.mat'
saveName = 'run_centroids_user_20232024_20192020ref_1band_v2.mat'
# saveName = 'run_centroids_user_20232024_1band.mat'
if doLogBandPower:      saveName = saveName.replace('centroids_user', 'centroids_logBandPower_user')
if not applyLaplacian:  saveName = saveName.replace('user', 'user_noLap')

# extract_run_centroids(covs_centered, cov_events, validRuns, classes, runVector, isCFeedback, runs_labels, saveData=True, pathData=pathData, doLogBandPower=doLogBandPower, saveName=saveName)

# import scripts.processing_centroids as proc_cent
# reload(proc_cent)
# saveName = 'day_centroids_user_20232024_20192020ref.mat' if not doLogBandPower else 'day_centroids_logBandPower_user_20232024_20192020ref.mat'
# if not applyLaplacian: saveName = saveName.replace('user', 'user_noLap')
# proc_cent.extract_day_centroids(covs_centered, cov_events, validRuns, classes, runVector, dayVector, isCFeedback, runs_labels, saveData=True, pathData=pathData, doLogBandPower=doLogBandPower, saveName=saveName)

In [ ]:
# filename = 'run_centroids_user_20232024.mat'
filename = 'run_centroids_user_20232024_20192020ref_1band_v2.mat'
if doLogBandPower:
    filename = filename.replace('centroids_user', 'centroids_logBandPower_user')
if not applyLaplacian:
    filename = filename.replace('user', 'user_noLap')


run_centroids = loadmat(f'{pathData}{filename}')

# ref_ForAngle = run_centroids['ref_angle']
d_ToEye = run_centroids['d_ToEye']
std_eyeDistance = run_centroids['std_eyeDistance']
std_centroids = run_centroids['std_centroids']
mAbsDev_centroids = run_centroids['mAbsDev_centroids']
run_centroids = run_centroids['run_centroids']


filename = 'run_centroids_user_1band_v2.mat'
if not applyLaplacian: filename = filename.replace('user', 'user_noLap')

ref_ForAngle = loadmat(f'{pathData}{filename}')['ref_angle']


In [ ]:
# from scipy.io import savemat
# matrix_angle = mrtrf.compute_runMatrix_angles(run_centroids)
# savemat(f'{pathData}matrix_angle_user2023.mat', {'matrix_angle': matrix_angle})


Computing angles for run 1/98


/home/palatella/workspace/Riemann-centroids-study/riemann_utils/matrix_functions.py:18: RuntimeWarning: invalid value encountered in arccos
  angle = np.arccos(cos_theta)


Computing angles for run 2/98
Computing angles for run 3/98
Computing angles for run 4/98
Computing angles for run 5/98
Computing angles for run 6/98
Computing angles for run 7/98
Computing angles for run 8/98
Computing angles for run 9/98
Computing angles for run 10/98
Computing angles for run 11/98
Computing angles for run 12/98
Computing angles for run 13/98
Computing angles for run 14/98
Computing angles for run 15/98
Computing angles for run 16/98
Computing angles for run 17/98
Computing angles for run 18/98
Computing angles for run 19/98
Computing angles for run 20/98
Computing angles for run 21/98
Computing angles for run 22/98
Computing angles for run 23/98
Computing angles for run 24/98
Computing angles for run 25/98
Computing angles for run 26/98
Computing angles for run 27/98
Computing angles for run 28/98
Computing angles for run 29/98
Computing angles for run 30/98
Computing angles for run 31/98
Computing angles for run 32/98
Computing angles for run 33/98
Computing angles

In [7]:
if doNormalize:     d_ToEye /= mAbsDev_centroids  # because it's between a distribution and a point

d_betweenClasses = np.full((2, run_centroids.shape[1], len(classes), len(classes)), np.nan)

for cl in range(len(classes)):
    for cl2 in range(len(classes)):
        if cl == cl2:
            d_betweenClasses[:,0,cl,cl2] = 0
            d_betweenClasses[:,1:,cl,cl2] = distance_riemann(run_centroids[:,1:,cl], run_centroids[:,:-1,cl2])
            if doNormalize:     d_betweenClasses[:,1:,cl,cl2] /= (mAbsDev_centroids[:,1:,cl]+mAbsDev_centroids[:,:-1,cl])/2
        else:
            d_betweenClasses[:,:,cl,cl2] = distance_riemann(run_centroids[:,:,cl], run_centroids[:,:,cl2])
            if doNormalize:     d_betweenClasses[:,:,cl,cl2] /= (mAbsDev_centroids[:,:,cl]+mAbsDev_centroids[:,:,cl2])/2

In [ ]:
center_point = np.eye(run_centroids.shape[-1])

angles_to_zero = np.full((run_centroids.shape[0], run_centroids.shape[1], len(classes)), np.nan)
for run_idx in np.ndindex(angles_to_zero.shape):
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = ref_ForAngle[run_idx[0]]
    vec = run_centroids[run_idx]
    angles_to_zero[run_idx[0], run_idx[1], run_idx[2]], _ = mrtrf.angle_between_matrices(base, vec, center_point)

# angles_to_zero = mrtrf.evaluate_negative_angles(angles_to_zero, run_centroids, center_point, positiveCen=run_centroids[0,0,0], positiveAngl=angles_to_zero[0,0,0])

angles_to_before = np.full(angles_to_zero.shape,  np.nan)
for run_idx in np.ndindex(angles_to_before.shape[:3]):
    if run_idx[1]>0:
        base = run_centroids[run_idx[0],run_idx[1]-1,run_idx[2]]
        vec = run_centroids[run_idx]
        angle, _ = mrtrf.angle_between_matrices(base, vec, center_point)
        angles_to_before[run_idx[0], run_idx[1], run_idx[2]] = angle
    else:  
        # saveName = 'run_centroids_user.mat' if not doLogBandPower else 'run_centroids_logBandPower_user.mat'
        # if not applyLaplacian: saveName = saveName.replace('user', 'user_noLap')
        # old_run_centroids = loadmat(f'/home/palatella/workspace/cybathlon_user/{saveName}')
        # base = old_run_centroids['run_centroids'][run_idx[0],-1,run_idx[2]]
        # vec = run_centroids[run_idx]
        # angle, _ = mrtrf.angle_between_matrices(base, vec, center_point)
        # angles_to_before[run_idx[0], run_idx[1], run_idx[2]] = angle
        angles_to_before[run_idx[0], run_idx[1], run_idx[2]] = 0


# angles_to_before = mrtrf.evaluate_negative_angles(angles_to_before, run_centroids, center_point, positiveCen=run_centroids[0,0,0], positiveAngl=angles_to_zero[0,0,0])

In [9]:
angles_to_otherClass = np.full((run_centroids.shape[0], run_centroids.shape[1]), np.nan)
for run_idx in np.ndindex(angles_to_otherClass.shape[:2]):
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = run_centroids[run_idx[0], run_idx[1], 0]
    vec = run_centroids[run_idx[0], run_idx[1], 1]
    angles_to_otherClass[run_idx[0], run_idx[1]], _ = mrtrf.angle_between_matrices(base, vec, center_point)

In [ ]:
# from scipy.io import savemat
# savemat(f'{pathData}completeResults_20232024_angles_ref20192020.mat', {'angles_to_otherClass': angles_to_otherClass, 'angles_to_before': angles_to_before, 'angles_to_zero': angles_to_zero, 'd_betweenClasses': d_betweenClasses, 'd_ToEye': d_ToEye, 'std_eyeDistance': std_eyeDistance, 'std_centroids': std_centroids, 'mAbsDev_centroids': mAbsDev_centroids})

In [ ]:
performances = loadmat(f'{pathData}raw_accuracy_user_rhlh')
run_accuracy = performances['raw_accuracy'][0]
run_rejection = performances['raw_rejection'][0]
x_accuracy = np.array(range(0, len(run_accuracy)))

performances = loadmat(f'{pathData}raw_accuracy_user.mat')

run_accuracy = np.concatenate((run_accuracy, performances['raw_accuracy'][0]))
run_rejection = np.concatenate((run_rejection, performances['raw_rejection'][0]))
x_accuracy = np.concatenate((x_accuracy, np.array(range(len(x_accuracy), len(run_accuracy)))))

In [11]:
rows = []
for nrun, band, task in np.ndindex((d_ToEye.shape[1], d_ToEye.shape[0], d_ToEye.shape[2])):
    rows.append({
        'nRun': nrun,
        'band': band,
        'task': task,
        'runId' : validRuns[nrun],
        'nDay': dayVector[run_starts[nrun]].astype(int),
        'dayLabel': dates[np.where(x_dates<=nrun)[0][-1]],

        'distance' : d_ToEye[band, nrun, task],
        'angle' : angles_to_zero[band, nrun, task],

        'btw_distance': d_betweenClasses[band, nrun, 0, 1],
        'btw_angle': angles_to_otherClass[band, nrun],

        'wth_distance': d_betweenClasses[band, nrun, task, task],
        'wth_angle': angles_to_before[band, nrun, task],
     
        'accuracy': run_accuracy[nrun],
        'rejection': run_rejection[nrun]
    })

resultsDf = pd.DataFrame(rows)

In [ ]:
resultsDf.to_csv(f'/home/palatella/workspace/cybathlon_results/results_user_2023_1band_v2.csv', index=False)


# BLOCK BY BLOCK

In [ ]:
n_samples = covs.shape[1]
n_days = np.max(cov_events['day'])

dayVector = utilsVector['day']-1
runVector = utilsVector['run']
protocolVector = utilsVector['protocol']
isCFeedback = utilsVector['isCFeedback'].astype(bool)

isCFeedback = isCFeedback & (protocolVector==2)  # only evaluations
validRuns = np.unique(runVector[isCFeedback]).astype(int)

## ---- Comment this part if fisher score and cva
discard_day = '03/10/2023'
n_day_discard = np.where(days == discard_day)[0]
n_run_discard = np.unique(runVector[dayVector==n_day_discard])

validRuns = np.setdiff1d(validRuns, n_run_discard)
## ------------------------------------------------
run_starts = np.unique(runVector, return_index=True)
idx_good_runs = np.intersect1d(run_starts[0], validRuns, return_indices=True)[1]
run_starts = run_starts[1][idx_good_runs]
dates_info = np.unique(dayVector[run_starts], return_index=True)
dates = days[dates_info[0].astype(int)]
x_dates = dates_info[1]


filenameMatrices = 'centering_logBandPower_matrices_20232024.mat' if doLogBandPower else 'centering_matrices_20232024.mat'
if not applyLaplacian: saveName = saveName.replace('matrices', 'matrices_noLap')

if os.path.exists(f'{pathData}{filenameMatrices}'):
    saveTransformMatrices = False
    matrices = loadmat(f'{pathData}{filenameMatrices}')
else:
    print('Creating new centering matrices...')
    saveTransformMatrices = True
    matrices = {}
    matrices['mean_cov'] = None
    matrices['inv_sqrt_mean_cov'] = None

# covs_centered, _, _ = center_covariances(covs, dayVector, isCFeedback, referenceDay=0, saveTransformMatrices=saveTransformMatrices, pathData=pathData, saveName=filenameMatrices, mean_cov=matrices['mean_cov'], inv_sqrt_mean_cov=matrices['inv_sqrt_mean_cov'])

saveName = 'run_centroids_user_20232024.mat'
if doLogBandPower:      saveName = saveName.replace('centroids_user', 'centroids_logBandPower_user')
if not applyLaplacian:  saveName = saveName.replace('user', 'user_noLap')

# extract_run_centroids(covs_centered, cov_events, validRuns, classes, runVector, isCFeedback, runs_labels, saveData=True, pathData=pathData, doLogBandPower=doLogBandPower, saveName=saveName)

In [ ]:
saveName = 'run_centroids_user_20232024.mat'
if doLogBandPower:      saveName = saveName.replace('centroids_user', 'centroids_logBandPower_user')
if not applyLaplacian:  saveName = saveName.replace('user', 'user_noLap')

run_centroids = loadmat(f'{pathData}{saveName}')

ref_ForAngle = run_centroids['ref_angle']
d_ToEye = run_centroids['d_ToEye']
std_eyeDistance = run_centroids['std_eyeDistance']
std_centroids = run_centroids['std_centroids']
mAbsDev_centroids = run_centroids['mAbsDev_centroids']
run_centroids = run_centroids['run_centroids']

In [46]:
if doNormalize:     d_ToEye /= mAbsDev_centroids  # because it's between a distribution and a point


d_betweenClasses = np.full((2, run_centroids.shape[1], len(classes), len(classes)), np.nan)

for cl in range(len(classes)):
    for cl2 in range(len(classes)):
        if cl == cl2:
            d_betweenClasses[:,0,cl,cl2] = 0
            d_betweenClasses[:,1:,cl,cl2] = distance_riemann(run_centroids[:,1:,cl], run_centroids[:,:-1,cl2])
            if doNormalize:     d_betweenClasses[:,1:,cl,cl2] /= (mAbsDev_centroids[:,1:,cl]+mAbsDev_centroids[:,:-1,cl])/2
        else:
            d_betweenClasses[:,:,cl,cl2] = distance_riemann(run_centroids[:,:,cl], run_centroids[:,:,cl2])
            if doNormalize:     d_betweenClasses[:,:,cl,cl2] /= (mAbsDev_centroids[:,:,cl]+mAbsDev_centroids[:,:,cl2])/2

In [47]:
center_point = np.eye(run_centroids.shape[-1])

angles_to_zero = np.full((run_centroids.shape[0], run_centroids.shape[1], len(classes)), np.nan)
classes_angles = np.full((run_centroids.shape[0], run_centroids.shape[1]), np.nan)
for run_idx in np.ndindex(angles_to_zero.shape):
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = ref_ForAngle[run_idx[0]]
    vec = run_centroids[run_idx]
    angles_to_zero[run_idx[0], run_idx[1], run_idx[2]], _ = mrtrf.angle_between_matrices(base, vec, center_point)

# angles_to_zero = mrtrf.evaluate_negative_angles(angles_to_zero, run_centroids, center_point, positiveCen=run_centroids[0,0,0], positiveAngl=angles_to_zero[0,0,0])
angles_to_otherClass = np.full((run_centroids.shape[0], run_centroids.shape[1]), np.nan)
for run_idx in np.ndindex(angles_to_otherClass.shape[:2]):
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = run_centroids[run_idx[0], run_idx[1], 0]
    vec = run_centroids[run_idx[0], run_idx[1], 1]
    angles_to_otherClass[run_idx[0], run_idx[1]], _ = mrtrf.angle_between_matrices(base, vec, center_point)
from scipy.io import savemat
# savemat(f'{pathData}completeResults_20192020_t_betwCl_angles.mat', {'angles_to_otherClass': angles_to_otherClass})

angles_to_before = np.zeros((run_centroids.shape[0], run_centroids.shape[1], len(classes)))
for run_idx in np.ndindex(angles_to_before.shape):
    if run_idx[1] == 0:
        continue
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = run_centroids[run_idx[0], run_idx[1]-1, run_idx[2]]
    vec = run_centroids[run_idx]
    angles_to_before[run_idx[0], run_idx[1], run_idx[2]], _ = mrtrf.angle_between_matrices(base, vec, center_point)
    
# from scipy.io import savemat
# savemat(f'{pathData}completeResults_20192020_t_before_angles.mat', {'angles_to_before': angles_to_before})

In [49]:
rows = []
for nrun, band, task in np.ndindex((d_ToEye.shape[1], d_ToEye.shape[0], d_ToEye.shape[2])):
    rows.append({
        # 'nRun': nrun,
        # 'band': band,
        # 'task': task,
        'runId' : validRuns[nrun],
        # 'nDay': dayVector[run_starts[nrun]].astype(int),
        # 'dayLabel': dates[np.where(x_dates<=nrun)[0][-1]],
        
        'distance' : d_ToEye[band, nrun, task],
        'angle' : angles_to_zero[band, nrun, task],

        # 'btw_distance': d_betweenClasses[band, nrun, 0, 1],
        'btw_angle': angles_to_otherClass[band, nrun],

        # 'wthin_distance': d_betweenClasses[band, nrun, task, task],
        'wth_angle': angles_to_before[band, nrun, task],
     
    })

blockResults1Df = pd.DataFrame(rows)

In [ ]:
idxBhBf = 79
winterRun = validRuns[idxBhBf]

saveTransformMatrices = True
matrices = {}
matrices['mean_cov'] = None
matrices['inv_sqrt_mean_cov'] = None
filenameMatrices = 'centering_logBandPower_matrices_bhbfCenter_20232024.mat' if doLogBandPower else 'centering_matrices_bhbfCenter_20232024.mat'
if not applyLaplacian: saveName = saveName.replace('matrices', 'matrices_noLap')

# covs_centered, _, _ = center_covariances(covs, dayVector, isCFeedback, referenceDay=dayVector[runVector>=winterRun][0], saveTransformMatrices=saveTransformMatrices, 
#                                          pathData=pathData, saveName=filenameMatrices, mean_cov=matrices['mean_cov'], inv_sqrt_mean_cov=matrices['inv_sqrt_mean_cov'])

saveName = 'run_centroids_user_bhbfCenter_20232024.mat' if not doLogBandPower else 'run_centroids_logBandPower_user_bhbfCenter_20232024.mat'
if not applyLaplacian: saveName = saveName.replace('user', 'user_noLap')

# extract_run_centroids(covs_centered, cov_events, validRuns, classes, runVector, isCFeedback, runs_labels, runRef=winterRun, saveData=True, pathData=pathData, doLogBandPower=doLogBandPower, saveName=saveName)

In [ ]:
saveName = 'run_centroids_user_bhbfCenter_20232024.mat' if not doLogBandPower else 'run_centroids_logBandPower_user_bhbfCenter_20232024.mat'
if not applyLaplacian: saveName = saveName.replace('user', 'user_noLap')

run_centroids = loadmat(f'{pathData}{saveName}')

ref_ForAngle = run_centroids['ref_angle']
d_ToEye = run_centroids['d_ToEye']
std_eyeDistance = run_centroids['std_eyeDistance']
std_centroids = run_centroids['std_centroids']
mAbsDev_centroids = run_centroids['mAbsDev_centroids']
run_centroids = run_centroids['run_centroids']

In [70]:
if doNormalize:     d_ToEye /= mAbsDev_centroids  # because it's between a distribution and a point


d_betweenClasses = np.full((2, run_centroids.shape[1], len(classes), len(classes)), np.nan)

for cl in range(len(classes)):
    for cl2 in range(len(classes)):
        if cl == cl2:
            d_betweenClasses[:,0,cl,cl2] = 0
            d_betweenClasses[:,1:,cl,cl2] = distance_riemann(run_centroids[:,1:,cl], run_centroids[:,:-1,cl2])
            if doNormalize:     d_betweenClasses[:,1:,cl,cl2] /= (mAbsDev_centroids[:,1:,cl]+mAbsDev_centroids[:,:-1,cl])/2
        else:
            d_betweenClasses[:,:,cl,cl2] = distance_riemann(run_centroids[:,:,cl], run_centroids[:,:,cl2])
            if doNormalize:     d_betweenClasses[:,:,cl,cl2] /= (mAbsDev_centroids[:,:,cl]+mAbsDev_centroids[:,:,cl2])/2

In [71]:
center_point = np.eye(run_centroids.shape[-1])

angles_to_zero = np.full((run_centroids.shape[0], run_centroids.shape[1], len(classes)), np.nan)
classes_angles = np.full((run_centroids.shape[0], run_centroids.shape[1]), np.nan)
for run_idx in np.ndindex(angles_to_zero.shape):
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = ref_ForAngle[run_idx[0]]
    vec = run_centroids[run_idx]
    angles_to_zero[run_idx[0], run_idx[1], run_idx[2]], _ = mrtrf.angle_between_matrices(base, vec, center_point)

# angles_to_zero = mrtrf.evaluate_negative_angles(angles_to_zero, run_centroids, center_point, positiveCen=run_centroids[0,0,0], positiveAngl=angles_to_zero[0,0,0])
angles_to_otherClass = np.full((run_centroids.shape[0], run_centroids.shape[1]), np.nan)
for run_idx in np.ndindex(angles_to_otherClass.shape[:2]):
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = run_centroids[run_idx[0], run_idx[1], 0]
    vec = run_centroids[run_idx[0], run_idx[1], 1]
    angles_to_otherClass[run_idx[0], run_idx[1]], _ = mrtrf.angle_between_matrices(base, vec, center_point)
from scipy.io import savemat
# savemat(f'{pathData}completeResults_20192020_t_betwCl_angles.mat', {'angles_to_otherClass': angles_to_otherClass})

angles_to_before = np.zeros((run_centroids.shape[0], run_centroids.shape[1], len(classes)))
for run_idx in np.ndindex(angles_to_before.shape):
    if run_idx[1] == 0:
        continue
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = run_centroids[run_idx[0], run_idx[1]-1, run_idx[2]]
    vec = run_centroids[run_idx]
    angles_to_before[run_idx[0], run_idx[1], run_idx[2]], _ = mrtrf.angle_between_matrices(base, vec, center_point)
    
# from scipy.io import savemat
# savemat(f'{pathData}completeResults_20192020_t_before_angles.mat', {'angles_to_before': angles_to_before})

In [72]:
rows = []
for nrun, band, task in np.ndindex((d_ToEye.shape[1], d_ToEye.shape[0], d_ToEye.shape[2])):
    rows.append({
        # 'nRun': nrun,
        # 'band': band,
        # 'task': task,
        'runId' : validRuns[nrun],
        # 'nDay': dayVector[run_starts[nrun]].astype(int),
        # 'dayLabel': dates[np.where(x_dates<=nrun)[0][-1]],
        
        'distance' : d_ToEye[band, nrun, task],
        'angle' : angles_to_zero[band, nrun, task],

        # 'btw_distance': d_betweenClasses[band, nrun, 0, 1],
        'btw_angle': angles_to_otherClass[band, nrun],

        # 'wthin_distance': d_betweenClasses[band, nrun, task, task],
        'wth_angle': angles_to_before[band, nrun, task],
     
    })

blockResults2Df = pd.DataFrame(rows)

In [78]:
copy1block = blockResults1Df.copy()
copy1block.pop('runId')
for col in copy1block.columns:
    copy1block.rename(columns={col: '(blk_'+col+')'}, inplace=True)
resultsDf = pd.concat([resultsDf, copy1block], axis=1)

In [ ]:
copy1block = blockResults1Df.copy()
copy1block = copy1block[copy1block.runId<winterRun]

copy2block = blockResults2Df.copy()
copy2block = copy2block[copy2block.runId>=winterRun]

winter = pd.merge(copy1block, copy2block, how='outer')
winter = winter[['distance', 'angle', 'btw_angle', 'wth_angle']]

for col in winter.columns:
    winter.rename(columns={col: 'blk_'+col}, inplace=True)

resultsDf = pd.concat([resultsDf, winter], axis=1)


In [ ]:
# saveName = 'results_user_2023.csv'
# resultsDf.to_csv(f"/home/palatella/workspace/cybathlon_results/{saveName}", index=False)

# FISHER SCORE AND CVA

In [ ]:
# savemat = 'psd_user_lhrh_20232024.mat'

# mat_data = loadmat(f'{pathData}{savemat}')
# psd = mat_data['psd']
# freq = mat_data['freq'].squeeze()
# psd_runVector = dtmn.fix_mat(mat_data['runVector']).squeeze()
# psd_protocolVector = dtmn.fix_mat(mat_data['protocolVector']).squeeze()
# channelBandDict = dtmn.fix_mat(mat_data['channelBandDict'])
# t_events = np.vectorize(lambda x: x[0][0])(mat_data['psd_events'])
# psd_events = pd.DataFrame(t_events, columns=[x[0] for x in mat_data['column_names'][0]])
# psd_events['isLhRh'] = 1

# savemat = 'psd_user_end2024.mat'
# mat_data = loadmat(f'{pathData}{savemat}')

# # isLhRh = np.concatenate((np.ones(psd.shape[0]), np.zeros(mat_data['psd'].shape[0]))).astype(bool)
# t_events = np.vectorize(lambda x: x[0][0])(mat_data['psd_events'])
# t_psd_events = pd.DataFrame(t_events, columns=[x[0] for x in mat_data['column_names'][0]])
# t_psd_events.pos += psd.shape[0]
# t_psd_events.day += psd_events['day'].iloc[-1]
# t_psd_events.run += psd_events['run'].iloc[-1]
# t_psd_events['isLhRh'] = 0
# psd_events = pd.concat((psd_events, t_psd_events), ignore_index=True)
# psd = np.concatenate((psd, mat_data['psd']), axis=0)
# psd_runVector = np.concatenate((psd_runVector, dtmn.fix_mat(mat_data['runVector']).squeeze()+psd_runVector[-1]), axis=0)
# psd_protocolVector = np.concatenate((psd_protocolVector, dtmn.fix_mat(mat_data['protocolVector']).squeeze()), axis=0)

In [18]:
# def lin_ccc(x, y):
#     """Compute Lin's Concordance Correlation Coefficient between two 1D arrays."""
#     x_mean = np.mean(x)
#     y_mean = np.mean(y)
#     s_x = np.var(x)
#     s_y = np.var(y)
#     cov_xy = np.cov(x, y, bias=True)[0, 1]
    
#     return (2 * cov_xy) / (s_x + s_y + (x_mean - y_mean)**2)


# fisher, cva = sgnpr.compute_fisher_score(psd, freq, classes, psd_runVector, runs_labels, isCFeedbackVector=isCFeedback, validRuns=validRuns)


# n_Runs = len(validRuns)

# ccc_scores_fisher = np.zeros(n_Runs)
# ccc_scores_cva = np.zeros(n_Runs)

# fisher_similarities = np.zeros(n_Runs)
# cva_similarities = np.zeros(n_Runs)

# for i in range(0,n_Runs):
#     if i == 0:
#         j = i
#     else:   
#         j = i - 1
#     ccc_scores_fisher[i] = lin_ccc(fisher[:, :, i].flatten(), fisher[:, :, j].flatten())
#     ccc_scores_cva[i] = lin_ccc(cva[:, :, i].flatten(), cva[:, :, j].flatten())

#     fisher_similarities[i] = np.corrcoef(fisher[:, :, i].flatten(), fisher[:, :, j].flatten())[0, 1]
#     cva_similarities[i] = np.corrcoef(cva[:, :, i].flatten(), cva[:, :, j].flatten())[0, 1]

In [19]:
# runId = 0
# completeDataFrame = pd.DataFrame(columns=['day', 'runId', 'radius', 'angle', 'task', 'band', 'accuracy', 'rejection', 'isLhRh'])

# for n_day,strDay in enumerate(dates):
#     totalRuns = x_dates[n_day+1]-x_dates[n_day] if n_day < len(dates)-1 else d_ToEye.shape[1]-x_dates[n_day]
#     for r in range(totalRuns):
#         for b in range(d_ToEye.shape[0]):
#             for t in range(d_ToEye.shape[2]):
#                 isLhRh = np.unique(psd_events.loc[psd_events['run']==validRuns[runId],'isLhRh'].values)
#                 if len(isLhRh) > 1:
#                     print(f'Warning: multiple isLhRh values for runId {runId}')
#                     isLhRh = isLhRh[0]
#                 completeDataFrame = pd.concat([completeDataFrame, pd.DataFrame({
#                     'day': [strDay],
#                     'runId': [runId],
#                     'radius': [d_ToEye[b, x_dates[n_day]+r, t]],
#                     'angle': [angles_to_zero[b, x_dates[n_day]+r, t]],
#                     'task': [t],
#                     'band': [b],
#                     'accuracy': [run_accuracy[x_dates[n_day]+r]],
#                     'rejection': [run_rejection[x_dates[n_day]+r]],
#                     'isLhRh': isLhRh
#                 })], ignore_index=True)
#         runId += 1

In [20]:
# cvaSimK = np.empty(cva_similarities.shape[0]*4)
# cvaCCCK = np.empty(cva_similarities.shape[0]*4)

# for k in range(0, cva_similarities.shape[0]):
#     cvaSimK[k*4:(k+1)*4] = cva_similarities[k]
#     cvaCCCK[k*4:(k+1)*4] = ccc_scores_cva[k]

# completeDataFrame['cvaSiml'] = cvaSimK
# completeDataFrame['cvaCCC'] = cvaCCCK

# saveName = 'completeResults_20232024.csv'
# # completeDataFrame.to_csv(f"{pathData}{saveName}", index=False)


# PLOTS

In [21]:
stop_training = ['14/10/2024']  
stop_idx = np.array([next((x_dates[i] for i, s in enumerate(dates) if s == pattern), -1) for pattern in stop_training])
recalibrations = ['11/10/2024']  # bhbf started again on 11/10/2024
rec_idx = np.array([next((x_dates[i] for i, s in enumerate(dates) if s == pattern), -1) for pattern in recalibrations])

idx_stop_plot = d_ToEye.shape[1]
idx_start_plot = 0

t_stop_idx = stop_idx[(stop_idx >= idx_start_plot) & (stop_idx < idx_stop_plot)] - idx_start_plot
t_rec_idx = rec_idx[(rec_idx >= idx_start_plot) & (rec_idx<idx_stop_plot)] - idx_start_plot
t_x_dates = x_dates[(x_dates >= idx_start_plot) & (x_dates<idx_stop_plot)] - idx_start_plot
t_x_accuracy = x_accuracy[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)] - idx_start_plot
                  

idx_stopRec = stop_idx[np.where((stop_idx >= idx_start_plot) & (stop_idx < idx_stop_plot))]-idx_start_plot
idx_recal = rec_idx[np.where((rec_idx >= idx_start_plot) & (rec_idx<idx_stop_plot))]-idx_start_plot

# std = std_centroids[:,idx_start_plot:idx_stop_plot]*5
# point_size=std,

saveFigures = False

In [ ]:
clss_dist = d_betweenClasses[:,idx_start_plot:idx_stop_plot, 0, 1]
withinClass_dist = np.diagonal(d_betweenClasses[:,idx_start_plot:idx_stop_plot], axis1=-2, axis2=-1)
# # # # from scipy.io import savemat
# # # # savemat(f'{pathData}completeResults_20232024_btwWith_clss_distance.mat', {'btwClss': clss_dist, 'withClss': withinClass_dist})

In [ ]:
distance = d_ToEye[:,idx_start_plot:idx_stop_plot]
angl = angles_to_otherClass[:,idx_start_plot:idx_stop_plot]

max_value = 1
max_angle = 2

plt.close('all')

fig = rplt.plot_centroids_movement(distance, classes, dates=dates[(x_dates >= idx_start_plot) & (x_dates < idx_stop_plot)], x_dates=t_x_dates, 
                                   accuracy=run_accuracy[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)], x_accuracy=t_x_accuracy, rejection=run_rejection[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)],
                                   max_value=max_value, title='Distance to center (eye matrix) over days', stop_idx=t_stop_idx, rec_idx=t_rec_idx, bandranges=bandranges)

if doLogBandPower:  filename = f'{pathImages}eyeDistance_logBandPower_20232024.svg'
else:               filename = f'{pathImages}eyeDistance_20232024.svg'
if doNormalize: filename = filename.replace('.svg', '_normalized.svg')
if not applyLaplacian: filename = filename.replace('.svg', '_noLap.svg')
if saveFigures:
    print('Saving figure to ' + filename)
    fig.savefig(filename, bbox_inches='tight', dpi=1000, format='svg')


fig = rplt.plot_centroids_movement(angl, classes, dates=dates[(x_dates >= idx_start_plot) & (x_dates<idx_stop_plot)], x_dates=t_x_dates, 
                                   accuracy=run_accuracy[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)], x_accuracy=t_x_accuracy, rejection=run_rejection[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)],
                                   max_value=max_angle, min_value=-0,   title='Centroids angles over days', stop_idx=t_stop_idx, rec_idx=t_rec_idx, bandranges=bandranges)

if doLogBandPower:  filename = f'{pathImages}angleVariation_logBandPower_20232024.svg'
else:               filename = f'{pathImages}angleVariation_20232024.svg'
if doNormalize: filename = filename.replace('.svg', '_normalized.svg')
if not applyLaplacian: filename = filename.replace('.svg', '_noLap.svg')
if saveFigures:
    print('Saving figure to ' + filename)
    fig.savefig(filename, bbox_inches='tight', dpi=1000, format='svg')


t = np.cos(angl)
fig = rplt.plot_centroids_movement(t, classes, dates=dates[(x_dates >= idx_start_plot) & (x_dates<idx_stop_plot)], x_dates=t_x_dates, 
                                   accuracy=run_accuracy[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)], x_accuracy=t_x_accuracy, rejection=run_rejection[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)],
                                   max_value=1, min_value=-1,   title='Centroids cosine angles over days', stop_idx=t_stop_idx, rec_idx=t_rec_idx, bandranges=bandranges)

filename = 'cos_angleVariation_20232024.svg' if not doLogBandPower else 'cos_angleVariation_logBandPower_20232024.svg'
if doNormalize: filename = filename.replace('.svg', '_normalized.svg')
if not applyLaplacian: filename = filename.replace('.svg', '_noLap.svg')
if saveFigures:
    print('Saving figure to ' + filename)
    fig.savefig(f'{pathImages}{filename}', bbox_inches='tight', dpi=1000, format='svg')

t = np.sin(angl)
fig = rplt.plot_centroids_movement(t, classes, dates=dates[(x_dates >= idx_start_plot) & (x_dates<idx_stop_plot)], x_dates=t_x_dates, 
                                   accuracy=run_accuracy[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)], x_accuracy=t_x_accuracy, rejection=run_rejection[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)],
                                   max_value=max_angle, min_value=0,   title='Centroids sine angles over days', stop_idx=t_stop_idx, rec_idx=t_rec_idx, bandranges=bandranges)

filename = 'sin_angleVariation_20232024.svg' if not doLogBandPower else 'sin_angleVariation_logBandPower_20232024.svg'
if doNormalize: filename = filename.replace('.svg', '_normalized.svg')
if not applyLaplacian: filename = filename.replace('.svg', '_noLap.svg')
if saveFigures:
    print('Saving figure to ' + filename)
    fig.savefig(f'{pathImages}{filename}', bbox_inches='tight', dpi=1000, format='svg')



fig = rplt.polarPlot_centroids(distance, angl, bandranges=bandranges, idx_stopRec=idx_stopRec, idx_recal=idx_recal, max_distance=max_value)

if doLogBandPower:  filename = f'{pathImages}polarPlot_logBandPower_20232024.svg'
else:               filename = f'{pathImages}polarPlot_20232024.svg'
if doNormalize: filename = filename.replace('.svg', '_normalized.svg')
if not applyLaplacian: filename = filename.replace('.svg', '_noLap.svg')
if saveFigures:
    print('Saving figure to ' + filename)
    fig.savefig(filename, bbox_inches='tight', dpi=1000, format='svg')